# Cobot Safety-State Detection - Full ML Pipeline

This notebook runs the end-to-end pipeline on the **full DASIG dataset (60 subjects)** using traditional Machine Learning models.

- Uses **0.5s windows** to better isolate abrupt events.
- Each ML model is in its own cell for isolated execution.
- Displays confusion matrices and detailed classification metrics inline for every model.

In [13]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

# Add the thesis directory to the python path to load our custom module
sys.path.insert(0, os.path.abspath('.'))

from cobot_safety_model.data_loader import load_all_trials, split_by_subject
from cobot_safety_model.features import segment_all_trials, extract_features_bulk, normalize_features
from cobot_safety_model.models import to_binary_labels, BINARY_CLASS_NAMES

from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay

## 1. Configuration & Data Loading

In [14]:
DATA_DIR = "data/multiphysio_hrc/DASIG"
WINDOW_SIZE = 0.5  # Reduced window size for better localization of abrupt events
STEP_SIZE = 0.25   # 50% overlap

print("Loading data for all 60 subjects...")
all_trials = load_all_trials(DATA_DIR, generate_labels=True, verbose=True)
print(f"Loaded {len(all_trials)} trials.")

Loading data for all 60 subjects...
  Loaded 30/180 trials...
  Loaded 60/180 trials...
  Loaded 90/180 trials...
  Loaded 120/180 trials...
  Loaded 150/180 trials...
  Done: 179/180 trials loaded successfully.
  Label distribution:
    SAFE:     3,068,961 samples (88.4%)
    DANGER:     358,000 samples (10.3%)
Loaded 179 trials.


## 2. Train/Val/Test Split

In [15]:
print("Splitting dataset by subject to prevent data leakage...")
train_trials, val_trials, test_trials = split_by_subject(
    all_trials, train_ratio=0.6, val_ratio=0.2, seed=42
)

Splitting dataset by subject to prevent data leakage...
Split: 36 train subjects (107 trials), 12 val subjects (36 trials), 12 test subjects (36 trials)


## 3. Sliding Window Segmentation

In [16]:
print(f"Segmenting trials into sliding windows (window={WINDOW_SIZE}s, step={STEP_SIZE}s)...")
# We use 'any_danger' label strategy
X_train_raw, y_train_raw = segment_all_trials(train_trials, WINDOW_SIZE, STEP_SIZE, label_strategy="any_danger")
X_val_raw, y_val_raw = segment_all_trials(val_trials, WINDOW_SIZE, STEP_SIZE, label_strategy="any_danger")
X_test_raw, y_test_raw = segment_all_trials(test_trials, WINDOW_SIZE, STEP_SIZE, label_strategy="any_danger")

print("\nSegmentation complete.")

Segmenting trials into sliding windows (window=0.5s, step=0.25s)...
Segmented 107 trials → 41302 windows (shape: (41302, 100, 65))
  SAFE:  35783 windows (86.6%)
  DANGER:   5136 windows (12.4%)
Segmented 36 trials → 13896 windows (shape: (13896, 100, 65))
  SAFE:  12039 windows (86.6%)
  DANGER:   1728 windows (12.4%)
Segmented 36 trials → 13896 windows (shape: (13896, 100, 65))
  SAFE:  12045 windows (86.7%)
  DANGER:   1728 windows (12.4%)

Segmentation complete.


## 4. Feature Extraction & Normalization

In [17]:
print("Extracting handcrafted features...")
X_train_feat, feature_names = extract_features_bulk(X_train_raw)
X_val_feat, _ = extract_features_bulk(X_val_raw, verbose=False)
X_test_feat, _ = extract_features_bulk(X_test_raw, verbose=False)

# Handle NaN/Inf values that might occur
for arr in [X_train_feat, X_val_feat, X_test_feat]:
    arr[~np.isfinite(arr)] = 0.0
    
print("\nNormalizing features...")
X_train_norm, X_val_norm, X_test_norm, (feat_mean, feat_std) = normalize_features(
    X_train_feat, X_val_feat, X_test_feat
)

print("Converting to binary classification (SAFE vs ABRUPT)...")
y_train = to_binary_labels(y_train_raw)
y_val = to_binary_labels(y_val_raw)
y_test = to_binary_labels(y_test_raw)
class_names = BINARY_CLASS_NAMES

print("Preprocessing complete! Ready for model training.")

Extracting handcrafted features...
Extracting 180 features from 41302 windows...
  Processed 1000/41302 windows...
  Processed 2000/41302 windows...
  Processed 3000/41302 windows...
  Processed 4000/41302 windows...
  Processed 5000/41302 windows...
  Processed 6000/41302 windows...
  Processed 7000/41302 windows...
  Processed 8000/41302 windows...
  Processed 9000/41302 windows...
  Processed 10000/41302 windows...
  Processed 11000/41302 windows...
  Processed 12000/41302 windows...
  Processed 13000/41302 windows...
  Processed 14000/41302 windows...
  Processed 15000/41302 windows...
  Processed 16000/41302 windows...
  Processed 17000/41302 windows...
  Processed 18000/41302 windows...
  Processed 19000/41302 windows...
  Processed 20000/41302 windows...
  Processed 21000/41302 windows...
  Processed 22000/41302 windows...
  Processed 23000/41302 windows...
  Processed 24000/41302 windows...
  Processed 25000/41302 windows...
  Processed 26000/41302 windows...
  Processed 27000/

## 5. Model Training & Evaluation

### Helper Function for Plotting Metrics

In [18]:
def evaluate_and_plot(model, X_test, y_test, class_names):
    y_pred = model.predict(X_test)
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names, digits=4))
    print(f"Macro F1 Score: {f1_score(y_test, y_pred, average='macro'):.4f}\n")
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, display_labels=class_names, cmap='Blues', ax=ax[0])
    ax[0].set_title('Confusion Matrix', fontweight='bold')
    
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, display_labels=class_names, cmap='Blues', normalize='true', values_format='.2%', ax=ax[1])
    ax[1].set_title('Normalized Confusion Matrix', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

### Model 1: Random Forest

In [19]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=300, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train_norm, y_train)

evaluate_and_plot(rf_model, X_test_norm, y_test, class_names)

Training Random Forest...
Classification Report:
              precision    recall  f1-score   support

        SAFE     0.9048    0.9913    0.9460     12045
      ABRUPT     0.8498    0.3209    0.4659      1851

    accuracy                         0.9020     13896
   macro avg     0.8773    0.6561    0.7060     13896
weighted avg     0.8974    0.9020    0.8821     13896

Macro F1 Score: 0.7060



### Model 2: Gradient Boosting

In [20]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_class_weight

print("Training Gradient Boosting...")
# Compute sample weights to handle class imbalance
weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
weight_map = dict(zip(np.unique(y_train), weights))
sample_weights = np.array([weight_map[y] for y in y_train])

gb_model = GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
gb_model.fit(X_train_norm, y_train, sample_weight=sample_weights)

evaluate_and_plot(gb_model, X_test_norm, y_test, class_names)

Training Gradient Boosting...
Classification Report:
              precision    recall  f1-score   support

        SAFE     0.9257    0.9241    0.9249     12045
      ABRUPT     0.5118    0.5176    0.5146      1851

    accuracy                         0.8700     13896
   macro avg     0.7187    0.7208    0.7198     13896
weighted avg     0.8706    0.8700    0.8703     13896

Macro F1 Score: 0.7198



### Model 3: XGBoost

In [21]:
import xgboost as xgb

print("Training XGBoost...")
num_safe = np.sum(y_train == 0)
num_abrupt = np.sum(y_train == 1)
scale_weight = num_safe / num_abrupt if num_abrupt > 0 else 1.0
print(f"Using scale_pos_weight: {scale_weight:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_norm, y_train)

evaluate_and_plot(xgb_model, X_test_norm, y_test, class_names)

Training XGBoost...
Using scale_pos_weight: 6.48
Classification Report:
              precision    recall  f1-score   support

        SAFE     0.9249    0.9370    0.9309     12045
      ABRUPT     0.5519    0.5051    0.5275      1851

    accuracy                         0.8795     13896
   macro avg     0.7384    0.7211    0.7292     13896
weighted avg     0.8752    0.8795    0.8772     13896

Macro F1 Score: 0.7292



### Model 4: Support Vector Machine (SVM)

In [22]:
from sklearn.svm import SVC

print("Training SVM...")
svm_model = SVC(kernel='rbf', class_weight='balanced', random_state=42, probability=True)
svm_model.fit(X_train_norm, y_train)

evaluate_and_plot(svm_model, X_test_norm, y_test, class_names)

Training SVM...
Classification Report:
              precision    recall  f1-score   support

        SAFE     0.9275    0.9059    0.9165     12045
      ABRUPT     0.4681    0.5392    0.5011      1851

    accuracy                         0.8570     13896
   macro avg     0.6978    0.7225    0.7088     13896
weighted avg     0.8663    0.8570    0.8612     13896

Macro F1 Score: 0.7088



### Model 5: Logistic Regression

In [23]:
from sklearn.linear_model import LogisticRegression

print("Training Logistic Regression...")
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train_norm, y_train)

evaluate_and_plot(lr_model, X_test_norm, y_test, class_names)

Training Logistic Regression...
Classification Report:
              precision    recall  f1-score   support

        SAFE     0.9269    0.8270    0.8741     12045
      ABRUPT     0.3384    0.5759    0.4263      1851

    accuracy                         0.7935     13896
   macro avg     0.6327    0.7014    0.6502     13896
weighted avg     0.8486    0.7935    0.8145     13896

Macro F1 Score: 0.6502



### Model 6: K-Nearest Neighbors (KNN)

In [24]:
from sklearn.neighbors import KNeighborsClassifier

print("Training K-Nearest Neighbors...")
# KNN does not have a built-in class_weight, so we fit it as is. 
# We can use weights='distance' to slightly favor closer points.
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
knn_model.fit(X_train_norm, y_train)

evaluate_and_plot(knn_model, X_test_norm, y_test, class_names)

Training K-Nearest Neighbors...
Classification Report:
              precision    recall  f1-score   support

        SAFE     0.9046    0.9866    0.9438     12045
      ABRUPT     0.7876    0.3225    0.4576      1851

    accuracy                         0.8982     13896
   macro avg     0.8461    0.6546    0.7007     13896
weighted avg     0.8890    0.8982    0.8791     13896

Macro F1 Score: 0.7007



## 6. Probability Threshold Tuning (Fixing False Negatives)
To hit a target recall of 90% for the ABRUPT class, we need to lower the classification threshold from the default 0.5. Here we plot the Precision-Recall curve and find the optimal threshold for XGBoost.

In [25]:
from sklearn.metrics import precision_recall_curve, confusion_matrix

print("Generating probabilities from XGBoost...")
y_probs = xgb_model.predict_proba(X_test_norm)[:, 1] # Probabilities for ABRUPT class

precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

# Find the threshold that guarantees at least 90% recall
target_recall = 0.90
idx = np.where(recall >= target_recall)[0][-1] # Last index where recall is >= 90%
optimal_threshold = thresholds[idx]

print(f"Target Recall: {target_recall*100}%")
print(f"Optimal Threshold: {optimal_threshold:.4f}")
print(f"Expected Precision at this threshold: {precision[idx]:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall[:-1], precision[:-1], color='blue', label='PR Curve')
ax.scatter(recall[idx], precision[idx], color='red', marker='o', s=100, label=f'Threshold = {optimal_threshold:.2f}')
ax.set_xlabel('Recall (Detecting ABRUPT)', fontweight='bold')
ax.set_ylabel('Precision (Avoiding False Alarms)', fontweight='bold')
ax.set_title('Precision-Recall Curve (XGBoost)', fontweight='bold')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.7)
plt.show()

print("\n--- Evaluation with New Threshold ---")
y_pred_tuned = (y_probs >= optimal_threshold).astype(int)
print(classification_report(y_test, y_pred_tuned, target_names=class_names, digits=4))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred_tuned)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', ax=ax[0])
ax[0].set_title('Confusion Matrix (Tuned Threshold)', fontweight='bold')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(cmap='Blues', values_format='.2%', ax=ax[1])
ax[1].set_title('Normalized Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

Generating probabilities from XGBoost...
Target Recall: 90.0%
Optimal Threshold: 0.1001
Expected Precision at this threshold: 0.1856

--- Evaluation with New Threshold ---
              precision    recall  f1-score   support

        SAFE     0.9624    0.3933    0.5584     12045
      ABRUPT     0.1856    0.9001    0.3078      1851

    accuracy                         0.4608     13896
   macro avg     0.5740    0.6467    0.4331     13896
weighted avg     0.8589    0.4608    0.5250     13896

